# Cross-Target Specificity Matrix
Fits each of the 6 locked models on its own training set, predicts on all 6 test sets.
Run all cells top to bottom. Output: `specificity_matrix.csv`

In [ ]:
!pip install einops rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 35.2 MB/s eta 0:00:00


In [ ]:
# Download ChEMBL v36 SQLite database
!wget -q --show-progress https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_36/chembl_36_sqlite.tar.gz

# Extract it
!tar -xzf chembl_36_sqlite.tar.gz

# Verify
!ls -lh chembl_36/chembl_36_sqlite/chembl_36.db

chembl_36_sqlite.ta 100%[===================>]   5.23G  46.5MB/s    in 1m 48s  
-rw-r--r-- 1 5012 1101 28G Sep 10  2025 chembl_36/chembl_36_sqlite/chembl_36.db


In [1]:
import sqlite3
import numpy as np
import pandas as pd
import torch, math
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.ML.Cluster import Butina
from sklearn.neural_network import MLPRegressor
from scipy.stats import pearsonr
from transformers import AutoTokenizer, AutoModel
print('Imports OK')

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
DB_PATH = 'chembl_36/chembl_36_sqlite/chembl_36.db'

TARGETS = {
    'GABRA1':  {'chembl_id': 'CHEMBL1962',  'arch': (512,256,128), 'act': 'tanh', 'alpha': 0.0001, 'lr': 0.0005},
    'CHRM4':   {'chembl_id': 'CHEMBL1821',  'arch': (512,256,128), 'act': 'relu', 'alpha': 0.01,   'lr': 0.001},
    'HCRTR1':  {'chembl_id': 'CHEMBL5113',  'arch': (512,256),     'act': 'tanh', 'alpha': 0.0001, 'lr': 0.001},
    'HCRTR2':  {'chembl_id': 'CHEMBL4792',  'arch': (512,256),     'act': 'tanh', 'alpha': 0.0001, 'lr': 0.001},
    'CHRM2':   {'chembl_id': 'CHEMBL211',   'arch': (512,256),     'act': 'relu', 'alpha': 0.001,  'lr': 0.0001},
    'ADORA1':  {'chembl_id': 'CHEMBL226',   'arch': (512,256,128), 'act': 'tanh', 'alpha': 0.001,  'lr': 0.001},
}

SEEDS = [42, 7, 13, 99, 21]
TARGET_NAMES = list(TARGETS.keys())
print('Config OK')

Config OK


In [3]:
print('Loading datasets from ChEMBL...')
conn = sqlite3.connect(DB_PATH)

SQL = '''
SELECT cs.canonical_smiles, act.pchembl_value
FROM activities act
JOIN assays a ON act.assay_id = a.assay_id
JOIN target_dictionary td ON a.tid = td.tid
JOIN compound_structures cs ON act.molregno = cs.molregno
WHERE td.chembl_id = '{chembl_id}'
  AND act.pchembl_value IS NOT NULL
  AND act.standard_relation = '='
'''

datasets = {}
for name, cfg in TARGETS.items():
    df = pd.read_sql_query(SQL.format(chembl_id=cfg['chembl_id']), conn)
    df = df.groupby('canonical_smiles')['pchembl_value'].median().reset_index()
    df = df[df['pchembl_value'] >= 5].reset_index(drop=True)
    datasets[name] = df
    print(f'  {name}: {len(df)} compounds')

conn.close()
print('Done.')

Loading datasets from ChEMBL...
  GABRA1: 357 compounds
  CHRM4: 1816 compounds
  HCRTR1: 7090 compounds
  HCRTR2: 7481 compounds
  CHRM2: 2316 compounds
  ADORA1: 4951 compounds
Done.


In [4]:
print('Loading MoLFormer-XL...')
tokenizer = AutoTokenizer.from_pretrained(
    'ibm-research/MoLFormer-XL-both-10pct', trust_remote_code=True)
molformer = AutoModel.from_pretrained(
    'ibm-research/MoLFormer-XL-both-10pct',
    deterministic_eval=True, trust_remote_code=True)
molformer.eval()
molformer.get_head_mask = lambda head_mask, num_layers, is_attention_chunked=False: [None] * num_layers

def _rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def _apply_rotary(q, k, cos, sin, pos_ids):
    cos = cos[pos_ids].unsqueeze(1)
    sin = sin[pos_ids].unsqueeze(1)
    return (q * cos) + (_rotate_half(q) * sin), (k * cos) + (_rotate_half(k) * sin)

def _fixed_attn_fwd(self, hidden_states, attention_mask=None, position_ids=None,
                    head_mask=None, output_attentions=False):
    q = self.transpose_for_scores(self.query(hidden_states))
    k = self.transpose_for_scores(self.key(hidden_states))
    v = self.transpose_for_scores(self.value(hidden_states))
    L = k.shape[-2]
    cos, sin = self.rotary_embeddings(v, seq_len=L)
    q, k = _apply_rotary(q, k, cos, sin, position_ids)
    q, k = self.feature_map(q, k)
    if attention_mask is not None:
        mask = (attention_mask == 0).to(attention_mask.dtype)
        per_pos = mask[:, 0, -1]
        k = k * per_pos[:, None, -L:, None]
    kv   = torch.matmul(k.transpose(-1, -2), v)
    norm = torch.matmul(q, k.sum(dim=-2).unsqueeze(-1)).clamp(min=self.eps)
    ctx  = torch.matmul(q, kv) / norm
    ctx  = ctx.permute(0, 2, 1, 3).contiguous()
    ctx  = ctx.view(ctx.size()[:-2] + (self.all_head_size,))
    return (ctx,)

import types
for layer in molformer.encoder.layer:
    layer.attention.self.forward = types.MethodType(_fixed_attn_fwd, layer.attention.self)
print('Patched all 12 attention layers')

def get_embedding(smiles):
    inputs = tokenizer(smiles, padding=True, return_tensors='pt')
    with torch.no_grad():
        outputs = molformer(**inputs)
    return outputs.pooler_output.squeeze().cpu().numpy()

print('MoLFormer ready.')

Loading MoLFormer-XL...
Patched all 12 attention layers
MoLFormer ready.


In [5]:
print('Generating embeddings (slow - ~10-20 min)...')
embeddings = {}
for name, df in datasets.items():
    print(f'  {name} ({len(df)} compounds)...')
    emb = np.array([get_embedding(s) for s in df['canonical_smiles']])
    embeddings[name] = emb
    print(f'    shape={emb.shape}  NaN={np.isnan(emb).any()}')
print('Embeddings done.')

Generating embeddings (slow - ~10-20 min)...
  GABRA1 (357 compounds)...
    shape=(357, 768)  NaN=False
  CHRM4 (1816 compounds)...
    shape=(1816, 768)  NaN=False
  HCRTR1 (7090 compounds)...
    shape=(7090, 768)  NaN=False
  HCRTR2 (7481 compounds)...
    shape=(7481, 768)  NaN=False
  CHRM2 (2316 compounds)...
    shape=(2316, 768)  NaN=False
  ADORA1 (4951 compounds)...
    shape=(4951, 768)  NaN=False
Embeddings done.


In [6]:
print('Running CCPart splits...')
splits = {}
for name, df in datasets.items():
    smiles_list = df['canonical_smiles'].tolist()
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))

    nfps = len(fps)
    dists = []
    for i in range(1, nfps):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1 - s for s in sims])

    clusters = Butina.ClusterData(dists, nfps, 0.6, isDistData=True)
    cluster_ids = np.zeros(nfps, dtype=int)
    for cid, cluster in enumerate(clusters):
        for idx in cluster:
            cluster_ids[idx] = cid

    unique_clusters = np.unique(cluster_ids)
    np.random.seed(42)
    np.random.shuffle(unique_clusters)
    n_test = int(0.2 * len(unique_clusters))
    test_clusters = set(unique_clusters[:n_test])

    train_idx = np.array([i for i in range(nfps) if cluster_ids[i] not in test_clusters])
    test_idx  = np.array([i for i in range(nfps) if cluster_ids[i] in test_clusters])
    splits[name] = (train_idx, test_idx)
    print(f'  {name}: train={len(train_idx)}  test={len(test_idx)}')
print('Splits done.')

Running CCPart splits...
  GABRA1: train=330  test=27
  CHRM4: train=1576  test=240
  HCRTR1: train=5898  test=1192
  HCRTR2: train=6082  test=1399
  CHRM2: train=1606  test=710
  ADORA1: train=3462  test=1489
Splits done.


In [7]:
print('Building 6x6 specificity matrix...')
matrix = np.zeros((6, 6))

for i, model_target in enumerate(TARGET_NAMES):
    cfg = TARGETS[model_target]
    train_idx, _ = splits[model_target]
    X_train = embeddings[model_target][train_idx]
    y_train = datasets[model_target]['pchembl_value'].values[train_idx]
    print(f'\nModel: {model_target} (train n={len(train_idx)})')

    for j, test_target in enumerate(TARGET_NAMES):
        _, test_idx = splits[test_target]
        X_test = embeddings[test_target][test_idx]
        y_test = datasets[test_target]['pchembl_value'].values[test_idx]

        rs = []
        for seed in SEEDS:
            mlp = MLPRegressor(
                hidden_layer_sizes=cfg['arch'],
                activation=cfg['act'],
                alpha=cfg['alpha'],
                learning_rate_init=cfg['lr'],
                max_iter=1000,
                early_stopping=True,
                validation_fraction=0.1,
                random_state=seed
            )
            mlp.fit(X_train, y_train)
            r, _ = pearsonr(mlp.predict(X_test), y_test)
            rs.append(r)

        mean_r = np.mean(rs)
        matrix[i, j] = mean_r
        tag = ' <-- diagonal' if i == j else ''
        print(f'  vs {test_target}: R={mean_r:.4f}  Std={np.std(rs):.4f}{tag}')

print('\nDone.')

Building 6x6 specificity matrix...

Model: GABRA1 (train n=330)
  vs GABRA1: R=0.6768  Std=0.0291 <-- diagonal
  vs CHRM4: R=-0.1589  Std=0.0489
  vs HCRTR1: R=-0.2400  Std=0.0250
  vs HCRTR2: R=-0.0864  Std=0.0903
  vs CHRM2: R=-0.3690  Std=0.0525
  vs ADORA1: R=-0.1017  Std=0.0294

Model: CHRM4 (train n=1576)
  vs GABRA1: R=-0.1499  Std=0.1270
  vs CHRM4: R=0.3486  Std=0.0417 <-- diagonal
  vs HCRTR1: R=0.1347  Std=0.0751
  vs HCRTR2: R=0.1025  Std=0.0466
  vs CHRM2: R=0.5696  Std=0.0128
  vs ADORA1: R=0.0231  Std=0.0620

Model: HCRTR1 (train n=5898)
  vs GABRA1: R=-0.5483  Std=0.1083
  vs CHRM4: R=0.1457  Std=0.0410
  vs HCRTR1: R=0.7279  Std=0.0069 <-- diagonal
  vs HCRTR2: R=0.1620  Std=0.0068
  vs CHRM2: R=0.4672  Std=0.0163
  vs ADORA1: R=-0.0704  Std=0.0131

Model: HCRTR2 (train n=6082)
  vs GABRA1: R=-0.4015  Std=0.0814
  vs CHRM4: R=-0.0306  Std=0.0474
  vs HCRTR1: R=-0.0114  Std=0.0238
  vs HCRTR2: R=0.6505  Std=0.0084 <-- diagonal
  vs CHRM2: R=0.3862  Std=0.0239
  vs ADORA

In [8]:
df_matrix = pd.DataFrame(matrix, index=TARGET_NAMES, columns=TARGET_NAMES)
df_matrix.index.name = 'Model vs Test'

print('6x6 Pearson R Specificity Matrix')
print('rows = model trained on | columns = test set predicted on')
print()
print(df_matrix.round(4).to_string())

df_matrix.to_csv('specificity_matrix.csv')
print('\nSaved to specificity_matrix.csv')

# Also download it directly
from google.colab import files
files.download('specificity_matrix.csv')

6x6 Pearson R Specificity Matrix
rows = model trained on | columns = test set predicted on

               GABRA1   CHRM4  HCRTR1  HCRTR2   CHRM2  ADORA1
Model vs Test                                                
GABRA1         0.6768 -0.1589 -0.2400 -0.0864 -0.3690 -0.1017
CHRM4         -0.1499  0.3486  0.1347  0.1025  0.5696  0.0231
HCRTR1        -0.5483  0.1457  0.7279  0.1620  0.4672 -0.0704
HCRTR2        -0.4015 -0.0306 -0.0114  0.6505  0.3862 -0.1302
CHRM2         -0.3902  0.5916  0.0156 -0.0586  0.6136  0.1285
ADORA1         0.0066  0.2778 -0.1171 -0.1078  0.2659  0.4007

Saved to specificity_matrix.csv


ModuleNotFoundError: No module named 'google.colab'